# Exploratory Data Analysis: Fantasy Football
## CSCI 5502 | Milestone 2

### Objective
Explore the 2025 NFL fantasy football player-week dataset, assess data quality and scoring distributions, compare ESPN projections with actual scores, and identify potential features for later modeling. This is descriptive EDA, not evidence that any particular feature improves future predictions.

**Dataset note:** This notebook uses the final 2025 dataset. All summaries below refer to the data loaded when the notebook runs. Missing values are preserved for initial exploration; analyses of projection error use only rows with both actual and projected scores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid")

# Supports running from the repository root or from notebooks/.
locations = [Path.cwd() / "data", Path.cwd().parent / "data"]
filenames = ["final_2025.csv", "df_final.csv"]
paths = [folder / name for folder in locations for name in filenames]
data_file = next((p for p in paths if p.is_file()), None)
if data_file is None:
    raise FileNotFoundError("Expected data/final_2025.csv or data/df_final.csv. Check the repository's data folder.")

df = pd.read_csv(data_file)
print("Loaded:", data_file)
print("Dataset shape:", df.shape)
display(df.head())

## 1. Dataset overview and descriptive statistics
Review the number of players and weeks, position balance, missingness, and standard summary statistics before interpreting the charts.

In [ ]:
print("Unique players:", df["player_id"].nunique())
print("Unique weeks:", df["week"].nunique())

print("\nRecords by position:")
display(df["position"].value_counts())

print("\nMissing values:")
display(df.isna().sum())

print("\nSummary statistics:")
display(df.describe())

In [ ]:
# Standard descriptive statistics not previously consolidated in EDATushar
print("Numeric summary (including quartiles):")
display(df.select_dtypes(include="number").describe().T.round(2))

print("Fantasy points: key statistics")
points = df["fantasy_points"]
summary = pd.Series({
    "count": points.count(), "mean": points.mean(), "median (Q2)": points.median(),
    "standard deviation": points.std(), "variance": points.var(),
    "min": points.min(), "Q1 (25%)": points.quantile(.25),
    "Q3 (75%)": points.quantile(.75), "max": points.max(),
    "IQR": points.quantile(.75)-points.quantile(.25),
    "skewness": points.skew()
})
display(summary.to_frame("value").round(2))

print("Missing values (nonzero only):")
missing = df.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({"missing": missing, "percent": (missing/len(df)*100).round(2)}).query("missing > 0"))

key = [x for x in ["player_id", "season", "week"] if x in df.columns]
print(f"Duplicate player-week keys ({key}):", df.duplicated(subset=key).sum())
print("Fully duplicated rows:", df.duplicated().sum())

### Interpretation
The numerical summary reports central tendency (mean and median), spread (standard deviation, quartiles and IQR), and skewness. Compare the mean and median to assess asymmetry, and inspect the IQR for variability. Missing-value counts apply to **all columns**, not just fantasy points. Duplicate player-week keys need review before modeling because repeated records can distort the training data. These diagnostics should be interpreted using the values produced by a fresh run.

## 2. Fantasy score categories

In [ ]:
df["score_category"] = np.select(
    [
        df["fantasy_points"].isna(),
        df["fantasy_points"] == 0,
        df["fantasy_points"] < 0,
        df["fantasy_points"] > 0
    ],
    [
        "Missing (NaN)",
        "Zero",
        "Negative",
        "Positive"
    ],
    default="Unknown"
)

# Count observations in each category
score_counts = df["score_category"].value_counts()

display(score_counts)

# Calculate percentages
score_percent = (
    df["score_category"]
    .value_counts(normalize=True) * 100
).round(2)

display(score_percent)

In [ ]:
plt.figure(figsize=(10, 5))

order = ["Missing (NaN)", "Zero", "Negative", "Positive"]

sns.countplot(
    data=df,
    x="score_category",
    order=order
)

plt.title("Fantasy Score Categories (2025)")
plt.xlabel("Score Category")
plt.ylabel("Number of Player-Week Records")

plt.tight_layout()
plt.show()

In [ ]:
# Count each score category
counts = df["score_category"].value_counts()

# Calculate percentage of total records
percentages = (
    df["score_category"].value_counts(normalize=True) * 100
).round(2)

# Combine into a table
summary = pd.DataFrame({
    "Count": counts,
    "Percentage (%)": percentages
})

display(summary)

### Interpretation

The dataset contains 7,572 player-week records from the 2025 NFL season. Approximately 71.87% of the records have positive fantasy scores, while 26.98% have scores of exactly zero. Only 1.15% of records have negative fantasy scores, and there are no missing values in the fantasy points column.

The relatively large proportion of zero scores suggests that some player-week observations have little or no recorded fantasy production. However, we cannot determine whether these players participated in games based on fantasy points alone.

Since there are no missing fantasy scores in this dataset, we do not need to handle missing values for this particular column. We will still investigate missing values in other features during preprocessing.

In [ ]:
# Count each score category by position
position_counts = pd.crosstab(
    df["position"],
    df["score_category"]
)

# Calculate percentages within each position
position_percent = (
    position_counts.div(
        position_counts.sum(axis=1),
        axis=0
    ) * 100
).round(2)

display(position_percent)

In [ ]:
print("Dataset shape:", df.shape)

print("\nMissing fantasy points:")
print(df["fantasy_points"].isna().sum())

print("\nScore categories:")
print(df["score_category"].value_counts(dropna=False))

print("\nPositions:")
print(df["position"].value_counts())

In [ ]:
print("Projection columns:")
print([col for col in df.columns if "project" in col.lower()])

print("\nMissing values:")
print(df[["fantasy_points", "projected_points"]].isna().sum())

print("\nSummary statistics:")
display(df[["fantasy_points", "projected_points"]].describe())


## 3. ESPN projected vs. actual fantasy points

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="projected_points",
    y="fantasy_points",
    alpha=0.3
)

# Perfect prediction reference line
plt.plot(
    [0, 60],
    [0, 60],
    color="red",
    linestyle="--",
    label="Perfect Prediction"
)

plt.title("Projected vs Actual Fantasy Points (2025)")
plt.xlabel("Projected Fantasy Points")
plt.ylabel("Actual Fantasy Points")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Compare records with available projections
comparison = df[df["projected_points"].notna()].copy()

# Actual minus projected points
comparison["error"] = (
    comparison["fantasy_points"] - comparison["projected_points"]
)

# Calculate metrics
mae = comparison["error"].abs().mean()
mean_error = comparison["error"].mean()

correlation = comparison[
    ["projected_points", "fantasy_points"]
].corr().iloc[0, 1]

print("Records compared:", len(comparison))
print("Mean Absolute Error:", round(mae, 2))
print("Mean Error:", round(mean_error, 2))
print("Correlation:", round(correlation, 3))

### Interpretation
The scatterplot shows a positive relationship between projected and actual fantasy points, with noticeable variation around the perfect-prediction line. The saved analysis found a correlation of **0.702**, a mean absolute error of **3.80** points and a mean signed error (actual minus projected) of **-0.18** points across **7,281** records with available projections. A negative mean signed error suggests slight average overprediction, but opposite-signed errors cancel, so it does **not** imply that individual predictions are usually close. The **291** missing projections are excluded from these error metrics. Rerun the notebook to verify these values against the current CSV.

## 4. Distribution of fantasy points by position

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="position",
    y="fantasy_points",
    order=["QB", "RB", "WR", "TE", "K"]
)

plt.title("Distribution of Fantasy Points by Position (2025)")
plt.xlabel("Player Position")
plt.ylabel("Actual Fantasy Points")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
position_stats = (
    df.groupby("position")["fantasy_points"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

display(position_stats)

### Visualization 3: Distribution of Fantasy Points by Position

The boxplot shows clear differences in fantasy scoring across player positions. Quarterbacks have the highest average and median fantasy points (13.11), while tight ends have the lowest median (1.70). Kickers have a median of 8.00 and a relatively smaller standard deviation of 4.80.

Running backs and wide receivers show right-skewed distributions, with means higher than their medians and several high-scoring outliers. Quarterbacks have the highest standard deviation (9.48), indicating substantial variation in their weekly scores.

These findings suggest that player position is an important feature for fantasy point prediction, and position-specific scoring patterns should be considered during modeling.

## 5. Weekly scoring trends

In [ ]:
print([col for col in df.columns if "week" in col.lower()])

In [ ]:
weekly_scores = (
    df.groupby("week")["fantasy_points"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

display(weekly_scores)

plt.figure(figsize=(11, 5))

sns.lineplot(
    data=weekly_scores,
    x="week",
    y="mean",
    marker="o",
    label="Mean Fantasy Points"
)

sns.lineplot(
    data=weekly_scores,
    x="week",
    y="median",
    marker="s",
    label="Median Fantasy Points"
)

plt.title("Average and Median Fantasy Points Across NFL Weeks (2025)")
plt.xlabel("NFL Week")
plt.ylabel("Fantasy Points")
plt.xticks(sorted(weekly_scores["week"].unique()))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Visualization 4: Average and Median Fantasy Points Across NFL Weeks

The line chart shows that average fantasy points remain relatively stable throughout the 2025 NFL season, generally ranging between 6 and 7 points per player-week. The highest average occurs around Week 5, while Week 18 has the lowest average.

The median fluctuates between approximately 3.3 and 4.6 points and remains consistently below the mean. This suggests that fantasy scoring is right-skewed, with some high-scoring performances increasing the overall average.

Although weekly scoring does not show a strong increasing or decreasing trend, there are noticeable fluctuations. This suggests that NFL week may be worth investigating as a potential feature, although changes in player participation and the composition of the dataset could also explain these patterns.

## 6. Projection error by position

In [ ]:
# Calculate absolute projection error for each player-week
comparison["absolute_error"] = comparison["error"].abs()

# Average absolute error by position
position_mae = (
    comparison.groupby("position")["absolute_error"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

display(position_mae)

plt.figure(figsize=(9, 5))

sns.barplot(
    data=position_mae,
    x="position",
    y="absolute_error",
    color="steelblue"
)

plt.title("ESPN Projection Error by Player Position (2025)")
plt.xlabel("Player Position")
plt.ylabel("Mean Absolute Error (Fantasy Points)")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

### Visualization 5: ESPN Projection Error by Player Position

The bar chart compares ESPN's Mean Absolute Error (MAE) across different player positions. Quarterbacks have the highest average prediction error at approximately 5.55 fantasy points, while tight ends have the lowest at approximately 2.91 points. Running backs, wide receivers, and kickers have relatively similar prediction errors, ranging from about 3.8 to 3.9 points.

These results suggest that ESPN's projection accuracy varies depending on player position. However, positions also have different scoring distributions, so higher absolute errors do not necessarily indicate worse relative prediction performance.

This finding is relevant to our project because incorporating position-specific features or evaluating models separately by position may help us better understand and improve fantasy point predictions.

## 7. Signed projection error distributions
This section adapts the raw-error histograms from `EDA.ipynb`. Unlike absolute error, signed error reveals the direction of ESPN projection bias. We use **projected minus actual** here: positive values indicate overprediction and negative values indicate underprediction.

In [ ]:
# Signed error: positive = overprediction, negative = underprediction
error_df = comparison.copy()
error_df["signed_error"] = error_df["projected_points"] - error_df["fantasy_points"]

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=error_df, x="signed_error", kde=True, stat="density", ax=ax)
mean_val = error_df["signed_error"].mean()
median_val = error_df["signed_error"].median()
ax.axvline(mean_val, color="red", linestyle="--", label=f"Mean: {mean_val:.2f}")
ax.axvline(median_val, color="green", linestyle="-.", label=f"Median: {median_val:.2f}")
ax.axvline(0, color="black", alpha=.5, linewidth=1)
ax.set(title="Distribution of ESPN projection errors (2025)", xlabel="Projected - actual fantasy points", ylabel="Density")
ax.legend()
plt.tight_layout()
plt.show()

display(error_df.groupby("position")["signed_error"].agg(["count", "mean", "median", "std"]).round(2))

In [ ]:
# Correctly filter the histogram and mean/median lines to each position.
positions = [p for p in ["QB", "RB", "WR", "TE", "K"] if p in error_df["position"].unique()]
fig, axes = plt.subplots(len(positions), 1, figsize=(10, 3.1 * len(positions)), sharex=True)
axes = np.atleast_1d(axes)
for ax, pos in zip(axes, positions):
    vals = error_df.loc[error_df["position"] == pos, "signed_error"].dropna()
    sns.histplot(vals, kde=True, stat="density", ax=ax)
    ax.axvline(vals.mean(), color="red", linestyle="--", label=f"Mean: {vals.mean():.2f}")
    ax.axvline(vals.median(), color="green", linestyle="-.", label=f"Median: {vals.median():.2f}")
    ax.axvline(0, color="black", alpha=.4)
    ax.set(title=f"Signed projection error: {pos}", xlabel="Projected - actual fantasy points", ylabel="Density")
    ax.legend()
plt.tight_layout()
plt.show()

### Interpretation
The signed-error plots supplement the MAE chart by showing **whether** projections tend to be too high or too low. Compare each position's mean and median to zero and note any long tails. The previous `EDA.ipynb` used the full dataset for every position histogram despite computing position-specific statistics; the revised plots correctly filter the data for each position. Avoid claiming a normal distribution or directional bias without examining the rerun results.

## 8. Boom and bust relative to ESPN projections

In [ ]:
comparison["performance_type"] = np.select(
    [
        comparison["error"] >= 5,
        comparison["error"] <= -5
    ],
    ["Boom", "Bust"],
    default="Near Projection"
)

boom_bust = pd.crosstab(
    comparison["position"],
    comparison["performance_type"],
    normalize="index"
) * 100

boom_bust = boom_bust.reindex(
    columns=["Boom", "Near Projection", "Bust"],
    fill_value=0
)

display(boom_bust.round(2))

boom_bust.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=["#2ca25f", "#9ecae1", "#de2d26"]
)

plt.title("Boom vs Bust Rates by Player Position (2025)")
plt.xlabel("Player Position")
plt.ylabel("Percentage of Player-Week Records")
plt.xticks(rotation=0)
plt.legend(title="Performance Type")
plt.tight_layout()
plt.show()

### Visualization 6: Boom vs. Bust Rates by Player Position

The stacked bar chart compares how frequently players outperform or underperform ESPN's fantasy projections by at least five points. Quarterbacks have the highest boom and bust rates, with approximately 20% of performances exceeding projections by five or more points and 24% falling below projections by at least five points.

Tight ends have the highest percentage of performances near their projections, at approximately 81%. Running backs and wide receivers show similar distributions, with most performances falling within five points of their projected scores.

These findings suggest that projection errors vary across positions. However, since quarterbacks typically score more points, a fixed five-point threshold may exaggerate their boom and bust rates relative to lower-scoring positions. Future analysis could use position-specific or normalized thresholds.

## 9. Home versus away performance

In [ ]:
location_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["home", "away", "location", "atvs", "venue"]
    )
]

print(location_columns)

for col in location_columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="position",
    y="fantasy_points",
    hue="atVs",
    order=["QB", "RB", "WR", "TE", "K"],
    hue_order=["home", "away"]
)

plt.title("Home vs Away Fantasy Performance by Position (2025)")
plt.xlabel("Player Position")
plt.ylabel("Actual Fantasy Points")
plt.legend(title="Game Location")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
home_away_stats = (
    df.groupby(["position", "atVs"])["fantasy_points"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

display(home_away_stats)

### Visualization 7: Home vs. Away Fantasy Performance by Position

The boxplot compares fantasy points scored during home and away games across five player positions. Overall, the scoring distributions are relatively similar, with substantial overlap between home and away performances.

Kickers show the largest difference in average fantasy points, scoring 8.60 points at home compared to 7.91 away. Quarterbacks, running backs, and wide receivers also have slightly higher home averages, while tight ends score marginally more away.

These results suggest that home-field advantage may have a small relationship with fantasy scoring, but the differences are not large enough to establish a strong effect from this analysis alone. Game location could still be useful as a predictive feature when combined with player statistics, opponent strength, and other matchup-related variables.

## 10. EDA conclusions and modeling implications

- **Baseline:** ESPN projections correlate with actual fantasy scores but still produce substantial player-week errors. Compare future models against this baseline using time-ordered validation.
- **Position:** Fantasy scoring distributions and projection errors differ by position, motivating position-aware features or evaluations.
- **History:** Weekly trends are descriptive and may reflect changing player participation. Use lagged statistics (not same-week outcomes) when creating predictive features.
- **Game context:** Home/away is worth testing, but the descriptive differences alone do not prove a predictive benefit.
- **Data quality:** Check missing projections, other missing columns, and duplicate player-week keys before training. Preserve an untouched copy of the source dataset.
- **Leakage:** Features such as final-season defensive rankings must not use information from games occurring after the prediction week.

**Scope:** These observations describe the 2025 dataset. The notebook does not establish causality or demonstrate that adding a feature improves out-of-sample accuracy.